In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, brier_score_loss

import time

project_root = Path(
    r"C:\Users\rajur\OneDrive\Desktop\next-move-ai"
)

feature_dir = (
    project_root
    / "data"
    / "processed"
    / "engineered_features"
    / "group_4_question_difficulty"
    / "base_features"
)

prediction_dir = (
    project_root
    / "data"
    / "processed"
    / "model_predictions"
)

prediction_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Feature directory:", feature_dir)
print("Prediction directory:", prediction_dir)

Project root: C:\Users\rajur\OneDrive\Desktop\next-move-ai
Feature directory: C:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\engineered_features\group_4_question_difficulty\base_features
Prediction directory: C:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions


In [2]:

fold_results = []
all_predictions = []

feature_columns = [
    "question_smoothed_accuracy",
    "log_question_previous_interactions"
]

target_column = "correct"

for fold_number in range(1, 6):

    print(f"\n{'='*45}")
    print(f"Training V4.2 — Fold {fold_number}")
    print(f"{'='*45}")

    # Load fold-specific data
    train_df = pd.read_csv(
        feature_dir / f"fold_{fold_number}_train.csv"
    )

    val_df = pd.read_csv(
        feature_dir / f"fold_{fold_number}_validation.csv"
    )

    X_train = train_df[feature_columns]
    y_train = train_df[target_column]

    X_val = val_df[feature_columns]
    y_val = val_df[target_column]

    # Build a fresh model for this fold
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(
            C=1.0,
            max_iter=1000,
            random_state=42
        ))
    ])

    # Train
    start_time = time.perf_counter()

    model.fit(X_train, y_train)

    training_time = time.perf_counter() - start_time

    # Predict probabilities
    start_time = time.perf_counter()

    y_pred_proba = model.predict_proba(X_val)[:, 1]

    prediction_time = time.perf_counter() - start_time

    # Evaluate
    fold_log_loss = log_loss(y_val, y_pred_proba)
    fold_brier = brier_score_loss(y_val, y_pred_proba)

    # Extract coefficients
    logistic_model = model.named_steps["logistic_regression"]

    coefficients = logistic_model.coef_[0]
    intercept = logistic_model.intercept_[0]

    fold_results.append({
        "fold": fold_number,
        "log_loss": fold_log_loss,
        "brier_score": fold_brier,
        "accuracy_coefficient": coefficients[0],
        "exposure_coefficient": coefficients[1],
        "intercept": intercept,
        "training_time": training_time,
        "prediction_time": prediction_time
    })

    # Store OOF predictions
    fold_predictions = pd.DataFrame({
        "user_id": val_df["user_id"].to_numpy(),
        "order_id": val_df["order_id"].to_numpy(),
        "fold": fold_number,
        "y_true": y_val.to_numpy(),
        "y_pred_proba": y_pred_proba,
        "model": "v4.2"
    })

    all_predictions.append(fold_predictions)

    print(f"Training interactions: {len(train_df)}")
    print(f"Validation interactions: {len(val_df)}")
    print(f"Log Loss: {fold_log_loss:.6f}")
    print(f"Brier Score: {fold_brier:.6f}")
    print(f"Accuracy coefficient: {coefficients[0]:.6f}")
    print(f"Exposure coefficient: {coefficients[1]:.6f}")
    print(f"Intercept: {intercept:.6f}")
    print(f"Training time: {training_time:.2f} seconds")
    print(f"Prediction time: {prediction_time:.2f} seconds")



Training V4.2 — Fold 1
Training interactions: 177324
Validation interactions: 42659
Log Loss: 0.618577
Brier Score: 0.214391
Accuracy coefficient: 0.488517
Exposure coefficient: 0.014717
Intercept: 0.695346
Training time: 0.11 seconds
Prediction time: 0.00 seconds

Training V4.2 — Fold 2
Training interactions: 176273
Validation interactions: 43710
Log Loss: 0.607889
Brier Score: 0.209315
Accuracy coefficient: 0.491121
Exposure coefficient: 0.009065
Intercept: 0.669487
Training time: 0.05 seconds
Prediction time: 0.00 seconds

Training V4.2 — Fold 3
Training interactions: 170186
Validation interactions: 49797
Log Loss: 0.626050
Brier Score: 0.217846
Accuracy coefficient: 0.483957
Exposure coefficient: 0.014570
Intercept: 0.710787
Training time: 0.05 seconds
Prediction time: 0.00 seconds

Training V4.2 — Fold 4
Training interactions: 178311
Validation interactions: 41672
Log Loss: 0.613704
Brier Score: 0.212132
Accuracy coefficient: 0.492644
Exposure coefficient: 0.007678
Intercept: 0.6

In [3]:
fold_results_df = pd.DataFrame(fold_results)

print("FOLD-WISE RESULTS")
print(fold_results_df.round(6).to_string(index=False))

print("\nAVERAGE CROSS-VALIDATION RESULTS")

print(
    "Average Log Loss:",
    fold_results_df["log_loss"].mean()
)

print(
    "Average Brier Score:",
    fold_results_df["brier_score"].mean()
)

print(
    "Average Accuracy Coefficient:",
    fold_results_df["accuracy_coefficient"].mean()
)

print(
    "Average Exposure Coefficient:",
    fold_results_df["exposure_coefficient"].mean()
)

FOLD-WISE RESULTS
 fold  log_loss  brier_score  accuracy_coefficient  exposure_coefficient  intercept  training_time  prediction_time
    1  0.618577     0.214391              0.488517              0.014717   0.695346       0.106215         0.002386
    2  0.607889     0.209315              0.491121              0.009065   0.669487       0.053917         0.003769
    3  0.626050     0.217846              0.483957              0.014570   0.710787       0.049371         0.002977
    4  0.613704     0.212132              0.492644              0.007678   0.683822       0.049629         0.002686
    5  0.608664     0.209779              0.484307              0.011343   0.683010       0.048979         0.002694

AVERAGE CROSS-VALIDATION RESULTS
Average Log Loss: 0.6149769455855203
Average Brier Score: 0.21269266497060219
Average Accuracy Coefficient: 0.4881091927556237
Average Exposure Coefficient: 0.011474614281314308


In [5]:
oof_predictions = pd.concat(
    all_predictions,
    ignore_index=True
)

prediction_path = prediction_dir / "v4_2_predictions.csv"

oof_predictions.to_csv(
    prediction_path,
    index=False
)

print("OOF prediction shape:", oof_predictions.shape)
print("Saved to:", prediction_path)
print("Missing values:", oof_predictions.isna().sum().sum())

print(
    "Duplicate interactions:",
    oof_predictions.duplicated(
        ["user_id", "order_id"]
    ).sum()
)

OOF prediction shape: (219983, 6)
Saved to: C:\Users\rajur\OneDrive\Desktop\next-move-ai\data\processed\model_predictions\v4_2_predictions.csv
Missing values: 0
Duplicate interactions: 0


V4.2 — Question Accuracy + Exposure

Features:
1. question_smoothed_accuracy
2. log_question_previous_interactions

Average CV Log Loss: 0.614977
Average CV Brier Score: 0.212693

Average Accuracy Coefficient: 0.488109
Average Exposure Coefficient: 0.011475

Model:
StandardScaler + LogisticRegression
C = 1.0
max_iter = 1000
random_state = 42